# H-N00 nnU-Net baseline

`H-N00.yaml`을 단일 설정으로 사용해 학습을 시작하고 `progress.json`에서 진행률, ETA, Dice, Recall, composite 및 early-stopping 상태를 표시합니다. 학습 출력은 `runs/nnunet/H-N00-launch.log`에도 저장됩니다.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path(r'C:\model')
DATA_ROOT = Path(r'C:\Datasets\hemorrhage\v1\model_ready\nnunet')
CONFIG_PATH = REPO_ROOT / 'configs/experiments/hemorrhage/H-N00.yaml'
RESULTS_ROOT = REPO_ROOT / 'runs/nnunet'

sys.path.insert(0, str(REPO_ROOT / 'src'))
from brainon_ai.nnunet.runner import expected_output_folder, write_launch_summary
from brainon_ai.nnunet.config import load_config

summary = write_launch_summary(CONFIG_PATH, DATA_ROOT, RESULTS_ROOT)
OUTPUT_FOLDER = expected_output_folder(load_config(CONFIG_PATH), RESULTS_ROOT)
PROGRESS_PATH = OUTPUT_FOLDER / 'progress.json'
summary

{'config': 'C:\\model\\configs\\experiments\\hemorrhage\\H-N00.yaml',
 'data_paths': {'nnUNet_raw': 'C:\\Datasets\\hemorrhage\\v1\\model_ready\\nnunet\\nnUNet_raw',
  'nnUNet_preprocessed': 'C:\\Datasets\\hemorrhage\\v1\\model_ready\\nnunet\\nnUNet_preprocessed',
  'nnUNet_results': 'C:\\model\\runs\\nnunet'},
 'output_folder': 'C:\\model\\runs\\nnunet\\Dataset001_ICHBinary\\nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres\\fold_0',
 'experiment': {'name': 'nnunet_baseline',
  'dataset_id': 1,
  'configuration': '3d_fullres',
  'fold': 0,
  'device': 'auto',
  'continue_training': False,
  'validation_checkpoint': 'best_composite'},
 'plans': {'base_identifier': 'nnUNetPlans',
  'output_identifier': 'ICHPlansBaselineV1',
  'patch_size': [20, 320, 256],
  'batch_size': 1},
 'trainer': {'epochs': 1000,
  'iterations_per_epoch': 250,
  'validation_iterations_per_epoch': 50,
  'save_every': 50,
  'learning_rate': 0.01,
  'weight_decay': 3e-05,
  'oversample_foreground_percent': 0.3

## 의존성 및 GPU 확인

이 셀이 실패하면 저장소 루트에서 `python -m pip install -r requirements-nnunet-cu130.txt`를 실행한 뒤 커널을 재시작합니다.

In [2]:
import torch
import nnunetv2

assert torch.cuda.is_available(), 'CUDA를 사용할 수 없습니다.'
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('nnU-Net:', getattr(nnunetv2, '__version__', '2.8.1'))
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
PyTorch: 2.13.0+cu130
nnU-Net: 2.8.1
CUDA: 13.0
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM (GB): 8.0


## 학습 시작

학습은 별도 프로세스로 실행됩니다. `continue_training: true`이면 기존 `checkpoint_final.pth`, `checkpoint_latest.pth`, `checkpoint_best.pth` 순으로 찾아 이어서 실행합니다.

In [3]:
import os
import subprocess

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = RESULTS_ROOT / 'H-N00-launch.log'
command = [
    sys.executable, '-m', 'brainon_ai.nnunet.runner',
    '--config', str(CONFIG_PATH),
    '--data-root', str(DATA_ROOT),
    '--results-root', str(RESULTS_ROOT),
]
child_env = os.environ.copy()
child_env['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + child_env.get('PYTHONPATH', '')
log_handle = LOG_PATH.open('a', encoding='utf-8')
process = subprocess.Popen(
    command, cwd=REPO_ROOT, env=child_env, stdout=log_handle, stderr=subprocess.STDOUT
)
log_handle.close()
print('PID:', process.pid)
print('Log:', LOG_PATH)
print('Progress:', PROGRESS_PATH)
print('Output:', OUTPUT_FOLDER)

PID: 12440
Log: C:\model\runs\nnunet\H-N00-launch.log
Progress: C:\model\runs\nnunet\Dataset001_ICHBinary\nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres\fold_0\progress.json
Output: C:\model\runs\nnunet\Dataset001_ICHBinary\nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres\fold_0


## 진행률 및 ETA 모니터링

이 셀을 중단해도 학습 프로세스는 계속 실행됩니다. 다시 실행하면 현재 상태부터 표시합니다.

In [4]:
import json
import time
from datetime import timedelta
from IPython.display import clear_output, display
from tqdm.auto import tqdm

terminal_states = {'completed', 'early_stopped', 'failed'}
bar = None
while True:
    if not PROGRESS_PATH.exists():
        clear_output(wait=True)
        print('trainer 초기화를 기다리는 중...')
        print('로그:', LOG_PATH)
        time.sleep(5)
        continue
    state = json.loads(PROGRESS_PATH.read_text(encoding='utf-8'))
    if bar is None:
        bar = tqdm(total=state['total_epochs'], desc='H-N00 epochs')
    bar.n = state['current_epoch']
    bar.set_postfix({
        'dice': state.get('pseudo_dice'),
        'recall': state.get('pseudo_recall'),
        'composite': state.get('composite'),
    })
    bar.refresh()
    eta = state.get('eta_seconds')
    print('status:', state['status'])
    print('epoch:', f"{state['current_epoch']}/{state['total_epochs']}")
    print('ETA:', str(timedelta(seconds=int(eta))) if eta else '계산 중')
    print('best dice / recall / composite:', state.get('best_dice'), state.get('best_recall'), state.get('best_composite'))
    print('early stopping:', f"{state.get('pruning_bad_checks', 0)}/{state.get('pruning_patience', 0)}")
    print('batch / patch:', state.get('batch_size'), state.get('patch_size'))
    if state['status'] in terminal_states:
        break
    time.sleep(10)

trainer 초기화를 기다리는 중...
로그: C:\model\runs\nnunet\H-N00-launch.log


H-N00 epochs:   0%|          | 0/1000 [00:00<?, ?it/s]

status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoch: 0/1000
ETA: 계산 중
best dice / recall / composite: None None None
early stopping: 0/30
batch / patch: 1 [20, 320, 256]
status: running
epoc

## 최근 로그 확인

In [ ]:
lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
print('\n'.join(lines[-80:]))